# NYC Crash Data — Exploration

**This notebook is for exploration only. It defines no analysis logic.**

Every transformation, feature and statistic used in the project lives in `src/`
and is imported here. If you want to change how something is computed, change it
in `src/` — not in a cell — so the scripts, the figures and the write-ups cannot
drift apart.

Run the pipeline first:

```bash
python src/download_data.py
python src/preprocess.py
python src/analysis.py
python src/visualize.py
```

Authoritative results: [`outputs/analysis_summary.md`](../outputs/analysis_summary.md)

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# All project logic comes from src/ -- nothing is redefined in this notebook.
from features import TIME_PERIOD_ORDER, DAY_NAME_ORDER, rate_table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

df = pd.read_parquet(ROOT / "data/processed/crashes_clean.parquet")
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(f"{df.crash_datetime.min()} -> {df.crash_datetime.max()}")

## 1. Sanity checks

Confirm the analysis window is what we think it is, and that 2026 is absent.

In [ ]:
print(df.year.value_counts().sort_index().to_string())
print("\n2026 rows present:", int((df.year == 2026).sum()))
print("duplicate collision_id:", int(df.collision_id.duplicated().sum()))
print(f"\ninjury crashes: {df.injury_crash.sum():,} ({df.injury_crash.mean() * 100:.2f}%)")
print(f"fatal crashes : {df.fatal_crash.sum():,} ({df.fatal_crash.mean() * 1000:.2f} per 1,000)")

## 2. Missingness

Borough is the field to watch: it is missing on ~30% of crashes, which is why
borough comparisons stay in the supporting-analysis tier and there is no map.

In [ ]:
pd.read_csv(ROOT / "outputs/summary_tables/missingness.csv")

## 3. The core question — frequency vs injury rate vs fatality rate

`rate_table` returns the rate together with its sample size and a Wilson 95%
interval. No rate in this project is ever looked at without its denominator.

In [ ]:
hourly = rate_table(df, "crash_hour", "injury_crash")
fatal = rate_table(df[~df.midnight_placeholder], "crash_hour", "fatal_crash")

view = hourly[["crash_hour", "n", "rate_pct", "ci_low_pct", "ci_high_pct"]].copy()
view["fatal_per_1k"] = fatal["rate"].values * 1000

print("peak crashes   :", int(view.loc[view.n.idxmax(), "crash_hour"]))
print("peak injury %  :", int(view.loc[view.rate_pct.idxmax(), "crash_hour"]))
print("peak fatal rate:", int(view.loc[view.fatal_per_1k.idxmax(), "crash_hour"]))
view.round(2)

## 4. The `00:00` placeholder

`crash_time` is never null, but exactly-midnight is recorded about twice as often
as any other clock minute. The giveaway is the fatal count, not the frequency:
a real hour of the night cannot contain a single fatal crash in 8,527 records.

In [ ]:
pd.read_csv(ROOT / "outputs/summary_tables/midnight_placeholder_sensitivity.csv").round(2)

## 5. Day x hour

Checking the smallest cell before trusting any cell.

In [ ]:
wh = rate_table(df, ["day_name", "crash_hour"], "injury_crash")
print("smallest cell n:", int(wh.n.min()), "| largest:", int(wh.n.max()))

wh.pivot(index="day_name", columns="crash_hour", values="rate_pct").reindex(DAY_NAME_ORDER).round(1)

## 6. Time periods and road users

Reminder: the road-user flags record **who was hurt**, not who was present.
Every `pedestrian_injury_crash` is by construction also an `injury_crash`, so
injury rates must never be compared *between* these flags.

In [ ]:
period = rate_table(df, "time_period", "injury_crash")[["time_period", "n", "rate_pct"]]
period["fatal_per_1k"] = rate_table(df, "time_period", "fatal_crash")["rate"].values * 1000
print(period.round(2).to_string(index=False))

for flag in ["pedestrian_injury_crash", "cyclist_injury_crash", "motorist_injury_crash"]:
    by_hour = rate_table(df, "crash_hour", flag)
    print(f"\n{flag}: {by_hour.rate_pct.min():.1f}% -> {by_hour.rate_pct.max():.1f}% of crashes")

## 7. Scratch space

Try things here. If something is worth keeping, move it into `src/analysis.py`
so it is regenerated by the pipeline and lands in the evidence base.